<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°03 - Miguel Ángel Nova Zúñiga


**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |


In [1]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)


**1.1 Dimensiones del dataset**

In [2]:
filas, columnas = df.shape
print(f'El dataset tiene {filas} filas y {columnas} columnas.')

El dataset tiene 8807 filas y 12 columnas.


**1.2 Tipos de datos por columna**

In [3]:
df.dtypes

,0
show_id,object
type,object
title,object
director,object
cast,object
country,object
date_added,object
release_year,int64
rating,object
duration,object


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


**1.3 Valores nulos por columna**

In [5]:
df.isna().sum().sort_values(ascending=False)

,0
director,2634
country,831
cast,825
date_added,10
rating,4
duration,3
show_id,0
type,0
title,0
release_year,0


**2. Transformar `date_added` a tipo fecha**

In [6]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['date_added'].dtype

dtype('<M8[ns]')

**3. Crear columnas auxiliares con `assign`**

In [7]:
df = df.assign(
    year_added=lambda x: x['date_added'].dt.year,
    month_added=lambda x: x['date_added'].dt.month
)
df[['title', 'date_added', 'year_added', 'month_added']].head()

,title,date_added,year_added,month_added
0,Dick Johnson Is Dead,2021-09-25,2021.0,9.0
1,Blood & Water,2021-09-24,2021.0,9.0
2,Ganglands,2021-09-24,2021.0,9.0
3,Jailbirds New Orleans,2021-09-24,2021.0,9.0
4,Kota Factory,2021-09-24,2021.0,9.0


## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

**4. Películas agregadas después de 2018 (con `.loc`)**

In [8]:
peliculas_post_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]
print(f'Se encontraron {len(peliculas_post_2018)} películas agregadas después de 2018.')
peliculas_post_2018[['title', 'year_added', 'rating', 'duration']].head()

Se encontraron 3701 películas agregadas después de 2018.


,title,year_added,rating,duration
0,Dick Johnson Is Dead,2021.0,PG-13,90 min
6,My Little Pony: A New Generation,2021.0,PG,91 min
7,Sankofa,2021.0,TV-MA,125 min
9,The Starling,2021.0,PG-13,104 min
12,Je Suis Karl,2021.0,TV-MA,127 min


**5.1 Filtrar títulos que contienen la palabra 'love' (case-insensitive)**

In [9]:
titulos_love = df.loc[df['title'].str.contains('love', case=False, na=False)]
print(f'Se encontraron {len(titulos_love)} títulos con la palabra "love".')
titulos_love[['title', 'type', 'release_year']].head(10)

Se encontraron 196 títulos con la palabra "love".


,title,type,release_year
25,Love on the Spectrum,TV Show,2021
158,Love Don't Cost a Thing,Movie,2003
159,Love in a Puff,Movie,2010
206,"LSD: Love, Sex Aur Dhokha",Movie,2010
227,Really Love,Movie,2020
246,Man in Love,Movie,2021
375,Resort to Love,Movie,2021
402,The Last Letter From Your Lover,Movie,2021
485,Lethal Love,Movie,2021
506,This Little Love Of Mine,Movie,2021


**5.2 Extraer la duración en minutos para las películas**

In [10]:
df = df.assign(
    duration_min=df.loc[df['type'] == 'Movie', 'duration']
                    .str.extract(r'(\d+)')
                    .astype(float)
)
df.loc[df['type'] == 'Movie', ['title', 'duration', 'duration_min']].head()

,title,duration,duration_min
0,Dick Johnson Is Dead,90 min,90.0
6,My Little Pony: A New Generation,91 min,91.0
7,Sankofa,125 min,125.0
9,The Starling,104 min,104.0
12,Je Suis Karl,127 min,127.0


**6. `explode()` sobre la columna `listed_in`**

In [11]:
df_generos = df.assign(
    listed_in=df['listed_in'].str.split(', ')
).explode('listed_in')

df_generos = df_generos.rename(columns={'listed_in': 'genre'})
df_generos[['title', 'genre']].head(10)

,title,genre
0,Dick Johnson Is Dead,Documentaries
1,Blood & Water,International TV Shows
1,Blood & Water,TV Dramas
1,Blood & Water,TV Mysteries
2,Ganglands,Crime TV Shows
2,Ganglands,International TV Shows
2,Ganglands,TV Action & Adventure
3,Jailbirds New Orleans,Docuseries
3,Jailbirds New Orleans,Reality TV
4,Kota Factory,International TV Shows


**7. Top 10 de géneros más frecuentes**

In [12]:
top10_generos = df_generos['genre'].value_counts().head(10)
top10_generos

,count
genre,
International Movies,2752
Dramas,2427
Comedies,1674
International TV Shows,1351
Documentaries,869
Action & Adventure,859
TV Dramas,763
Independent Movies,756
Children & Family Movies,641


**8. `where()` y `mask()` para marcar películas largas (>120 min)**

In [13]:
# Usando mask: marca como 'Largo' las películas con más de 120 minutos
df['contenido_largo_mask'] = pd.Series('Corto', index=df.index).mask(
    df['duration_min'] > 120, 'Largo'
)

# Usando where: complemento; conserva 'Largo' donde se cumple, sino 'Corto'
df['contenido_largo_where'] = pd.Series('Largo', index=df.index).where(
    df['duration_min'] > 120, 'Corto'
)

df.loc[df['type'] == 'Movie', ['title', 'duration_min', 'contenido_largo_mask', 'contenido_largo_where']].head(10)

,title,duration_min,contenido_largo_mask,contenido_largo_where
0,Dick Johnson Is Dead,90.0,Corto,Corto
6,My Little Pony: A New Generation,91.0,Corto,Corto
7,Sankofa,125.0,Largo,Largo
9,The Starling,104.0,Corto,Corto
12,Je Suis Karl,127.0,Largo,Largo
13,Confessions of an Invisible Girl,91.0,Corto,Corto
16,Europe's Most Dangerous Man: Otto Skorzeny in ...,67.0,Corto,Corto
18,Intrusion,94.0,Corto,Corto
22,Avvai Shanmughi,161.0,Largo,Largo
23,Go! Go! Cory Carson: Chrissy Takes the Wheel,61.0,Corto,Corto


**9. Películas con duración >100 min, rating 'R' y país 'United States'**

In [14]:
filtro_pelis = df.loc[
    (df['type'] == 'Movie') &
    (df['duration_min'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]
print(f'Se encontraron {len(filtro_pelis)} películas que cumplen los criterios.')
filtro_pelis[['title', 'duration_min', 'rating', 'country', 'release_year']].head(10)

Se encontraron 243 películas que cumplen los criterios.


,title,duration_min,rating,country,release_year
48,Training Day,122.0,R,United States,2001
81,Kate,106.0,R,United States,2021
131,Blade Runner: The Final Cut,117.0,R,United States,1982
139,Do the Right Thing,120.0,R,United States,1989
144,House Party,104.0,R,United States,1990
174,Tears of the Sun,121.0,R,United States,2003
175,The Blue Lagoon,105.0,R,United States,1980
178,The Interview,112.0,R,United States,2014
183,In the Line of Fire,129.0,R,United States,1993
247,Sweet Girl,110.0,R,United States,2021


**10. Top 10 películas más largas con `.style`**

In [15]:
top10_largas = (
    df.loc[df['type'] == 'Movie', ['title', 'country', 'release_year', 'duration_min']]
      .sort_values('duration_min', ascending=False)
      .head(10)
      .reset_index(drop=True)
)

(top10_largas.style
    .background_gradient(subset=['duration_min'], cmap='Reds')
    .format({'duration_min': '{:.0f} min', 'release_year': '{:.0f}'})
    .set_caption('Top 10 películas más largas en Netflix')
    .set_properties(**{'text-align': 'left'})
)

,title,country,release_year,duration_min
0,Black Mirror: Bandersnatch,United States,2018,312 min
1,Headspace: Unwind Your Mind,nan,2021,273 min
2,The School of Mischief,Egypt,1973,253 min
3,No Longer kids,Egypt,1979,237 min
4,Lock Your Girls In,nan,1982,233 min
5,Raya and Sakina,nan,1984,230 min
6,Once Upon a Time in America,"Italy, United States",1984,229 min
7,Sangam,India,1964,228 min
8,Lagaan,"India, United Kingdom",2001,224 min
9,Jodhaa Akbar,India,2008,214 min


### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?


**11. Combinaciones más frecuentes de género y rating**

In [16]:
combinaciones = (
    df_generos[['genre', 'rating']]
      .value_counts()
      .head(10)
      .reset_index(name='conteo')
)
combinaciones

,genre,rating,conteo
0,International Movies,TV-MA,1130
1,International Movies,TV-14,1065
2,Dramas,TV-MA,830
3,International TV Shows,TV-MA,714
4,Dramas,TV-14,693
5,International TV Shows,TV-14,472
6,Comedies,TV-14,465
7,TV Dramas,TV-MA,434
8,Comedies,TV-MA,431
9,Dramas,R,375


**12. Películas con mismo título y distinto año de lanzamiento**

In [17]:
titulos_repetidos = (
    df.loc[df['type'] == 'Movie', ['title', 'release_year']]
      .drop_duplicates()
)
duplicados = titulos_repetidos.loc[titulos_repetidos['title'].duplicated(keep=False)]
duplicados_ord = duplicados.sort_values(['title', 'release_year'])
print(f'Existen {duplicados["title"].nunique()} títulos repetidos con distinto año de lanzamiento.')
duplicados_ord.head(20)

Existen 0 títulos repetidos con distinto año de lanzamiento.


,title,release_year


**13. Cantidad de títulos únicos**

In [18]:
n_unicos = df['title'].nunique()
print(f'Hay {n_unicos} títulos únicos en la columna title.')
print(f'Total de filas en el dataset: {len(df)}')
print(f'Diferencia (posibles duplicados): {len(df) - n_unicos}')

Hay 8807 títulos únicos en la columna title.
Total de filas en el dataset: 8807
Diferencia (posibles duplicados): 0
